# Etapa 3 – Coleta das séries brutas

Este notebook baixa cada série da fonte original e salva o arquivo **sem nenhuma alteração** em `dados/brutos/AAAA-MM-DD/` (data do download). Ao final, grava um `manifesto.csv` com fonte, URL, período coberto, número de observações e hash SHA-256 de cada arquivo, para que a base possa ser auditada e reproduzida.

- Fontes: Federal Reserve (H.15), Cboe, Yahoo Finance, ANBIMA e Banco Central. O FRED foi substituído pelas fontes primárias das mesmas séries em 21/10/2026, quando estava em manutenção.
- Decisões e códigos das séries: `plano-de-trabalho.md`, seções 3.1 e 3.2.
- O que cada fonte representa na análise: `codigo/resumo.md`.
- Para rodar: abrir o notebook a partir da pasta `codigo/` ou da raiz do repositório e executar todas as células (*Run All*). Dependências em `requirements.txt`.

In [1]:
import csv
import hashlib
import io
import json
import re
import time
import zipfile
from datetime import date, datetime
from pathlib import Path

import pandas as pd
import requests
import yfinance as yf

# Raiz do repositório: funciona tanto abrindo o notebook em codigo/ quanto na raiz.
RAIZ = Path.cwd().parent if Path.cwd().name == "codigo" else Path.cwd()
PASTA_BRUTOS = RAIZ / "dados" / "brutos" / date.today().isoformat()

# Início da coleta: um ano antes do início da amostra (set. 2003), para que a
# janela de 252 dias do drawdown já esteja completa no primeiro dia da amostra.
INICIO_COLETA = "2002-09-01"

USER_AGENT = "Mozilla/5.0 (artigo-flight-to-safety; coleta academica)"
TENTATIVAS = 3

## Funções auxiliares

- `baixar`: faz o download com até 3 tentativas. Aceita uma função de validação, usada quando a fonte responde "200 OK" com conteúdo inválido (acontece no SGS e em sites em manutenção).
- `salvar`: grava o conteúdo, sem alterar, na pasta do dia.
- `registro`: monta a linha do manifesto de cada arquivo.

In [2]:
def baixar(url, validar=None):
    """GET com novas tentativas; devolve o conteúdo em bytes.

    validar: função opcional aplicada ao conteúdo; se levantar ValueError, a
    resposta é tratada como falha e o download é repetido.
    """
    for tentativa in range(1, TENTATIVAS + 1):
        try:
            resposta = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=60)
            resposta.raise_for_status()
            if validar:
                validar(resposta.content)
            return resposta.content
        except (requests.RequestException, ValueError) as erro:
            if tentativa == TENTATIVAS:
                raise
            print(f"    tentativa {tentativa} falhou ({erro}); repetindo...")
            time.sleep(5 * tentativa)


def salvar(nome, conteudo):
    caminho = PASTA_BRUTOS / nome
    caminho.write_bytes(conteudo)
    return caminho


def registro(arquivo, fonte, serie, descricao, url, primeira, ultima, linhas):
    return {
        "arquivo": arquivo.name,
        "fonte": fonte,
        "serie": serie,
        "descricao": descricao,
        "url": url,
        "primeira_data": primeira,
        "ultima_data": ultima,
        "observacoes": linhas,
        "sha256": hashlib.sha256(arquivo.read_bytes()).hexdigest(),
        "data_download": date.today().isoformat(),
    }

## 1. Federal Reserve, publicação H.15: Treasury 10 anos e TIPS 10 anos

O H.15 (*Selected Interest Rates*) é a fonte primária das séries DGS10 e DFII10 do FRED. O pacote pronto do Data Download Program só traz os títulos nominais, então a coleta baixa o **pacote completo da publicação** (zip com um XML no padrão SDMX), que contém as duas séries usadas:

- `RIFLGFCY10_N.B`: yield nominal do Treasury de 10 anos, vencimento constante (equivale ao DGS10);
- `RIFLGFCY10_XII_N.B`: yield real do TIPS de 10 anos, vencimento constante (equivale ao DFII10).

O zip é salvo inteiro, sem alteração; a extração das duas séries fica para o processamento. Aqui só se confere que o arquivo é um zip válido e que as duas séries estão nele.

In [3]:
URL_H15 = "https://www.federalreserve.gov/datadownload/Output.aspx?rel=H15&filetype=zip"
SERIES_H15 = {
    "RIFLGFCY10_N.B": "Treasury 10 anos, yield nominal (% a.a.); equivale ao DGS10",
    "RIFLGFCY10_XII_N.B": "TIPS 10 anos, yield real (% a.a.); equivale ao DFII10",
}


def validar_zip(conteudo):
    try:
        zipfile.ZipFile(io.BytesIO(conteudo)).testzip()
    except zipfile.BadZipFile as erro:
        raise ValueError(f"o H.15 não devolveu um zip válido: {erro}")


def datas_serie_h15(xml, serie):
    """Primeira e última data e número de observações de uma série do XML do H.15."""
    trecho = xml.split(f'SERIES_NAME="{serie}"', 1)[1].split("</kf:Series>", 1)[0]
    datas = re.findall(r'OBS_VALUE="[0-9.]+" TIME_PERIOD="([0-9-]+)"', trecho)
    return datas[0], datas[-1], len(datas)


def coletar_h15():
    conteudo = baixar(URL_H15, validar=validar_zip)
    arquivo = salvar("h15_completo.zip", conteudo)
    with zipfile.ZipFile(arquivo) as z:
        xml = z.read("H15_data.xml").decode()
    registros = []
    for serie, descricao in SERIES_H15.items():
        if f'SERIES_NAME="{serie}"' not in xml:
            raise RuntimeError(f"Série {serie} não encontrada no pacote do H.15")
        primeira, ultima, n = datas_serie_h15(xml, serie)
        registros.append(registro(arquivo, "Federal Reserve (H.15)", serie, descricao,
                                  URL_H15, primeira, ultima, n))
    return registros

## 2. Cboe: VIX

A Cboe calcula e publica o VIX; é a fonte primária da série VIXCLS do FRED (que corresponde à coluna `CLOSE`). O arquivo traz abertura, máxima, mínima e fechamento desde 1990, com datas no formato americano (mm/dd/aaaa).

In [4]:
URL_CBOE_VIX = "https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv"


def validar_csv_cboe(conteudo):
    if not conteudo.startswith(b"DATE,OPEN,HIGH,LOW,CLOSE"):
        raise ValueError("arquivo da Cboe sem o cabeçalho esperado")


def coletar_cboe():
    conteudo = baixar(URL_CBOE_VIX, validar=validar_csv_cboe)
    arquivo = salvar("cboe_vix.csv", conteudo)
    linhas = conteudo.decode().strip().splitlines()[1:]
    primeira = datetime.strptime(linhas[0].split(",")[0], "%m/%d/%Y").date().isoformat()
    ultima = datetime.strptime(linhas[-1].split(",")[0], "%m/%d/%Y").date().isoformat()
    return [registro(arquivo, "Cboe", "VIX", "VIX, abertura/máxima/mínima/fechamento",
                     URL_CBOE_VIX, primeira, ultima, len(linhas))]

## 3. Yahoo Finance

S&P 500 (`^GSPC`), Ibovespa (`^BVSP`) o ETF IEF (Treasuries de 7 a 10 anos, usado para conferir a conversão do yield de 10 anos em retorno) e o câmbio USD/BRL de fechamento (`BRL=X`, robustez da PTAX na P3). As fontes primárias dos índices são a S&P Dow Jones Indices e a B3.

O parâmetro `end` é exclusivo: o dia corrente fica de fora, porque o valor dele ainda é intradiário.

In [5]:
SERIES_YAHOO = {
    "^GSPC": "S&P 500, índice de preço",
    "^BVSP": "Ibovespa",
    "IEF": "ETF iShares 7-10 Year Treasury (conferência do yield de 10 anos)",
    "BRL=X": "USD/BRL, câmbio de fechamento (robustez da PTAX)",
}


def coletar_yahoo():
    registros = []
    for ticker, descricao in SERIES_YAHOO.items():
        dados = yf.download(ticker, start=INICIO_COLETA, end=date.today().isoformat(),
                            auto_adjust=False, progress=False, multi_level_index=False)
        if dados.empty:
            raise RuntimeError(f"Yahoo Finance não devolveu dados para {ticker}")
        nome = ticker.replace("^", "").replace("=", "").lower()
        arquivo = PASTA_BRUTOS / f"yahoo_{nome}.csv"
        dados.to_csv(arquivo)
        registros.append(registro(arquivo, "Yahoo Finance", ticker, descricao,
                                  f"https://finance.yahoo.com/quote/{ticker}",
                                  dados.index.min().date().isoformat(),
                                  dados.index.max().date().isoformat(), len(dados)))
    return registros

## 4. ANBIMA

Índices de títulos públicos brasileiros: IMA-B (série principal da NTN-B), IMA-B 5+, IDkA IPCA 10 anos e IRF-M. A ANBIMA Data publica um arquivo com o histórico completo de cada índice, então não é preciso baixar dia a dia.

O arquivo vem com extensão `.xls`, mas o conteúdo é `.xlsx`; por isso é salvo como `.xlsx`.

In [6]:
URL_ANBIMA = (
    "https://s3-data-prd-use1-precos.s3.us-east-1.amazonaws.com"
    "/arquivos/indices-historico/{codigo}-HISTORICO.xls"
)
SERIES_ANBIMA = {
    "IMAB": "IMA-B (série principal da NTN-B)",
    "IMAB5MAIS": "IMA-B 5+ (robustez)",
    "IDKAIPCA10A": "IDkA IPCA 10 anos (robustez; desde dez. 2005)",
    "IRFM": "IRF-M (robustez, prefixados)",
}


def coletar_anbima():
    registros = []
    for codigo, descricao in SERIES_ANBIMA.items():
        url = URL_ANBIMA.format(codigo=codigo)
        arquivo = salvar(f"anbima_{codigo}.xlsx", baixar(url))
        datas = pd.read_excel(arquivo, usecols=["Data de Referência"])["Data de Referência"]
        registros.append(registro(arquivo, "ANBIMA", codigo, descricao, url,
                                  datas.min().date().isoformat(),
                                  datas.max().date().isoformat(), len(datas)))
    return registros

## 5. Banco Central do Brasil (SGS)

Taxa de câmbio PTAX de venda (série 1). A API limita consultas diárias a 10 anos por chamada, então o período é baixado em blocos e as observações são juntadas num único JSON, sem alterar os valores.

A API às vezes responde "200 OK" com corpo vazio ou erro 502; nesses casos o download é repetido.

In [7]:
URL_SGS = (
    "https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
    "?formato=json&dataInicial={inicio}&dataFinal={fim}"
)
SERIES_SGS = {
    1: "USD/BRL, PTAX venda (R$/US$)",
}


def data_iso(data_sgs):
    """Converte dd/mm/aaaa (formato do SGS) para aaaa-mm-dd, só para o manifesto."""
    dia, mes, ano = data_sgs.split("/")
    return f"{ano}-{mes}-{dia}"


def coletar_sgs():
    registros = []
    hoje = date.today()
    for codigo, descricao in SERIES_SGS.items():
        observacoes = []
        inicio = date.fromisoformat(INICIO_COLETA)
        while inicio <= hoje:
            fim = min(date(inicio.year + 9, 12, 31), hoje)
            url = URL_SGS.format(codigo=codigo, inicio=inicio.strftime("%d/%m/%Y"),
                                 fim=fim.strftime("%d/%m/%Y"))
            observacoes.extend(json.loads(baixar(url, validar=json.loads)))
            inicio = date(fim.year + 1, 1, 1)
        arquivo = salvar(f"sgs_{codigo}.json",
                         json.dumps(observacoes, ensure_ascii=False, indent=0).encode())
        registros.append(registro(arquivo, "Banco Central (SGS)", str(codigo), descricao,
                                  URL_SGS.format(codigo=codigo, inicio="...", fim="..."),
                                  data_iso(observacoes[0]["data"]),
                                  data_iso(observacoes[-1]["data"]), len(observacoes)))
    return registros

## 6. Execução e manifesto

Roda as cinco coletas. Se uma fonte falhar (por exemplo, um site em manutenção), as outras continuam, e a fonte com falha é listada no final. Basta rodar o notebook de novo mais tarde: os arquivos do dia são sobrescritos.

In [8]:
PASTA_BRUTOS.mkdir(parents=True, exist_ok=True)
print(f"Salvando em {PASTA_BRUTOS.relative_to(RAIZ)}")

registros, falhas = [], []
for nome, funcao in [("Federal Reserve (H.15)", coletar_h15), ("Cboe", coletar_cboe),
                     ("Yahoo Finance", coletar_yahoo),
                     ("ANBIMA", coletar_anbima), ("Banco Central", coletar_sgs)]:
    print(f"- {nome}")
    try:
        novos = funcao()
    except Exception as erro:
        print(f"    FALHOU: {erro}")
        falhas.append(nome)
        continue
    for r in novos:
        print(f"    {r['serie']:<12} {r['primeira_data']} a {r['ultima_data']} ({r['observacoes']} obs.)")
    registros.extend(novos)

with open(PASTA_BRUTOS / "manifesto.csv", "w", newline="", encoding="utf-8") as f:
    escritor = csv.DictWriter(f, fieldnames=list(registros[0]))
    escritor.writeheader()
    escritor.writerows(registros)

if falhas:
    print(f"\nFontes com falha: {', '.join(falhas)}. Rode o notebook de novo mais tarde.")
else:
    print("\nColeta concluída.")

Salvando em dados/brutos/2026-10-21
- Federal Reserve (H.15)


    RIFLGFCY10_N.B 1962-01-02 a 2026-10-20 (16185 obs.)
    RIFLGFCY10_XII_N.B 2003-01-02 a 2026-10-20 (4999 obs.)
- Cboe


    VIX          1990-01-02 a 2026-10-20 (9298 obs.)
- Yahoo Finance


    ^GSPC        2002-09-03 a 2026-10-20 (6072 obs.)
    ^BVSP        2002-09-02 a 2026-10-20 (5982 obs.)
    IEF          2002-09-03 a 2026-10-20 (6072 obs.)
    BRL=X        2003-12-01 a 2026-10-20 (5520 obs.)
- ANBIMA


/Users/marinaladeira/Documents/artigo-flight-to-safety-eua-brasil/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


/Users/marinaladeira/Documents/artigo-flight-to-safety-eua-brasil/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


/Users/marinaladeira/Documents/artigo-flight-to-safety-eua-brasil/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


/Users/marinaladeira/Documents/artigo-flight-to-safety-eua-brasil/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


    IMAB         2003-09-16 a 2026-10-21 (5802 obs.)
    IMAB5MAIS    2003-09-16 a 2026-10-21 (5802 obs.)
    IDKAIPCA10A  2005-12-30 a 2026-10-21 (5224 obs.)
    IRFM         2000-02-01 a 2026-10-21 (6707 obs.)
- Banco Central


    1            2002-09-02 a 2026-10-21 (6064 obs.)

Coleta concluída.


In [9]:
pd.read_csv(PASTA_BRUTOS / "manifesto.csv")[
    ["fonte", "serie", "descricao", "primeira_data", "ultima_data", "observacoes"]
]

,fonte,serie,descricao,primeira_data,ultima_data,observacoes
0,Federal Reserve (H.15),RIFLGFCY10_N.B,"Treasury 10 anos, yield nominal (% a.a.); equi...",1962-01-02,2026-10-20,16185
1,Federal Reserve (H.15),RIFLGFCY10_XII_N.B,"TIPS 10 anos, yield real (% a.a.); equivale ao...",2003-01-02,2026-10-20,4999
2,Cboe,VIX,"VIX, abertura/máxima/mínima/fechamento",1990-01-02,2026-10-20,9298
3,Yahoo Finance,^GSPC,"S&P 500, índice de preço",2002-09-03,2026-10-20,6072
4,Yahoo Finance,^BVSP,Ibovespa,2002-09-02,2026-10-20,5982
5,Yahoo Finance,IEF,ETF iShares 7-10 Year Treasury (conferência do...,2002-09-03,2026-10-20,6072
6,Yahoo Finance,BRL=X,"USD/BRL, câmbio de fechamento (robustez da PTAX)",2003-12-01,2026-10-20,5520
7,ANBIMA,IMAB,IMA-B (série principal da NTN-B),2003-09-16,2026-10-21,5802
8,ANBIMA,IMAB5MAIS,IMA-B 5+ (robustez),2003-09-16,2026-10-21,5802
9,ANBIMA,IDKAIPCA10A,IDkA IPCA 10 anos (robustez; desde dez. 2005),2005-12-30,2026-10-21,5224
